In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import expm
from itertools import product

In [7]:
def sum_Hinnerprod(v):
    L = np.array([1, 0])
    R = np.array([0, 1])
    
    a = 1
    mu = 4
    H = np.array([[a+mu,-a/2,-a/2,0],
                      [-a/2,a,0,-a/2],
                      [-a/2,0,a,-a/2],
                      [0,-a/2,-a/2,a+mu]])
    
    # state corresponding to the equation of motion
    v = v
    
    # by setting s as a sum of L and R we implicitly sum over j,k,l=L,R
    s = L + R
    bra_sum1 = np.kron(v, s)
    bra_sum2 = np.kron(s, v)
    ket_sum = np.kron(s, s)
    
    total = psi_g2j.conj()*(bra_sum1.conj() @ H @ ket_sum)
    
    print(total)
    return total

In [6]:
total = sum_Hinnerprod(R)

4.0


In [ ]:
def psi_rhs(t, f, psi, H):
    """
    Return dpsi/dt.

    Parameters:
    t : float
    f : complex ndarray, shape (N,)
        Values f_gamma(t).
    psi : complex ndarray, shape (N, 2, 2)
        psi[gamma, particle, state].
    H : complex ndarray, shape (4, 4)

    Returns
    dpsi : complex ndarray, shape (N, 2, 2)
    """

    dpsi = np.zeros_like(psi, dtype=complex)

    for gamma in range(N): # number of slaters
        for n in range(2): # number of particles
            for v in range(2): # number of single-particle basis states L,R
                
                dpsi[gamma, n, u] = -1j * psi[gamma, n, u]

    return dpsi

In [10]:
# number of Slater determinants
N = 2

H = np.array([[a+mu,-a/2,-a/2,0],
              [-a/2,a,0,-a/2],
              [-a/2,0,a,-a/2],
              [0,-a/2,-a/2,a+mu]], dtype=complex)

def pair_index(a,b):
    # to get the index in the 2-particle basis LL LR RL RR
    return 2*a+b

def psi_rhs(t,f,psi,H,singular_tolerange=1e-12):
    """
    Here we define the equations for d_psi/dt, for all gamma, n (particle), and v (state L,R)

    Conventions:
    
    f.shape = (N,)
        f[gamma] = f_gamma(t)

    psi.shape = (N,2,2)
        psi[gamma, 0, 0] = psi_(gamma,1,L)
        psi[gamma, 0, 1] = psi_(gamma,1,R)
        psi[gamma, 1, 0] = psi_(gamma,2,L)
        psi[gamma, 1, 1] = psi_(gamma,2,R)

    H.shape = (4,4)
        Basis LL LR RL RR

    Returns
    d_psi : complex ndarray with shape (N,2,2)
        d_psi[gamma, 0, 0] = d_psi_(gamma,1,L)/dt
    """
    
    # number of slaters
    N = len(f)

    # The equation contains 1/f_gamma.
    if np.any(np.abs(f) < singular_tolerance):
        bad_indices = np.where(np.abs(f) < singular_tolerance)[0]

        raise FloatingPointError(
            "The psi equation is singular because the following "
            f"f_gamma values are close to zero: gamma={bad_indices.tolist()}. "
            "The equation explicitly contains 1/f_gamma."    )

    # Make 2-particle ket for H |psi_beta,1  psi_beta,2 > where 1 and 2 are the particles 
    # in basis LL LR RL RR
    psi12_beta = np.einsum("bk,bl->bkl", psi[:, 0, :], psi[:, 1, :]).reshape(N,4)
    f_psi12_beta = f[:,np.newaxis] * psi12_beta 
    # above, the newaxis allows us to multiply componentwise, so that f_beta for each beta is multiplied by all 4 entries of psi12 for that beta
    f_psi12 = np.sum(f_psi12_beta,axis=0) # sum over beta
    # now apply the Hamiltonian:
    f_H_psi12 = H @ f_psi12


    # Now start making the equations of motion
    dpsi = np.zeros_like(psi, dtype=complex)

    for gamma in range(N): # for the possible Slaters
        for v in range(2): # for 2 states L,R of the psi on LHS of equation
            
            # now we need 2 separate forms depending on if the equation is for the 1st particle or the second particle
            
            # 1st particle: index=0
            total_n1 = 0.0j

            for j in range(2): # possible L,R of the other particle
                vj_index = pair_index(v,j)
                total_n1 += ( np.conj(psi[gamma,1,j]) * f_H_psi12[vj_index] )
    